# Terraform Generator avec Callbacks LangChain Natifs

Ce notebook démontre l'utilisation des **callbacks LangChain** pour tracker les phases d'exécution de manière native et élégante.

## Approche

Au lieu d'un wrapper superficiel, on utilise le système de callbacks natif de LangChain :
- **`on_tool_start`** : Détecte le début d'une phase (planning, validation, review)
- **`on_tool_end`** : Capture le résultat et le succès
- **`on_llm_start`** : Détecte la génération de code

## Avantages

✅ **Natif LangChain** : Utilise le système de callbacks standard
✅ **Non intrusif** : Pas de modification du workflow existant
✅ **Phases réelles** : Basées sur les appels d'outils, pas sur du parsing
✅ **Structuré** : Logs et rapports JSON propres
✅ **Extensible** : Facile d'ajouter des checks personnalisés

## Étape 1: Imports

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import json

# Load environment variables
load_dotenv()

# Setup path
import sys
sys.path.insert(0, str(Path.cwd().parent))

from terraform_agent import (
    Config,
    PromptManager,
    KnowledgeBase,
    TerraformGenerator,
    TerraformPhaseCallback,        # Simple phase tracking
    DetailedTerraformCallback,      # With security/BP checks
)

print("✓ Imports loaded successfully")

## Étape 2: Configuration

In [ ]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)

print("✓ Logging configured")

## Étape 3: Initialisation des Composants

In [ ]:
# Get project root
project_root = Path.cwd().parent

# Initialize components
config = Config(base_dir=project_root)
print(f"Project Root: {config.PROJECT_ROOT}")
print(f"Work Dir: {config.WORK_DIR}")

prompts = PromptManager(config)
print("\n✓ Prompts loaded")

knowledge_base = KnowledgeBase(config)

# Initialize generator (standard)
generator = TerraformGenerator(config, prompts, knowledge_base)

print("\n✓ Generator ready")

## Étape 4: Exécution avec Callback Simple

Le callback `TerraformPhaseCallback` détecte automatiquement les phases :
- **PLANNING** : Quand `search_knowledge_base` ou `load_module_spec` est appelé
- **GENERATION** : Quand le LLM est invoqué
- **VALIDATION** : Quand `terraform_init`, `terraform_validate` ou `terraform_plan` est appelé
- **CODE_REVIEW** : Quand `review_and_fix_code` est appelé

In [ ]:
# Create callback
callback = TerraformPhaseCallback(verbose=True)

# Load user prompt
user_prompt = (project_root / "user_prompts/1-bucket.md").read_text()

# Execute with callback
print("\n🚀 Starting Terraform generation with phase tracking...\n")
result = generator.run(user_prompt=user_prompt, callbacks=[callback])

print("\n✓ Execution completed")

## Étape 5: Récupérer le Rapport

Le callback génère un rapport JSON structuré avec :
- Log des phases (nom, durée, timestamp)
- Résultats des outils (succès/échec)
- Durée totale

In [ ]:
# Get structured report
report = callback.get_report()

print("\n📊 Execution Report:")
print(json.dumps(report, indent=2))

## Étape 6: Exécution avec Callback Détaillé

Le callback `DetailedTerraformCallback` ajoute :
- **Security checks** : UBLA, encryption, public access, versioning, lifecycle
- **Best practices checks** : Module structure, variables, outputs, documentation
- **Scoring** : Security score et BP score (0-100%)

In [ ]:
# Create detailed callback
detailed_callback = DetailedTerraformCallback(verbose=True)

# Execute with detailed callback
print("\n🚀 Starting Terraform generation with detailed tracking...\n")
result = generator.run(user_prompt=user_prompt, callbacks=[detailed_callback])

print("\n✓ Execution completed")

## Étape 7: Rapport Détaillé avec Scores

In [ ]:
# Get detailed report with security/BP checks
detailed_report = detailed_callback.get_report()

print("\n📊 Detailed Execution Report:")
print(json.dumps(detailed_report, indent=2))

# Extract scores
if "security_score" in detailed_report:
    print(f"\n🔒 Security Score: {detailed_report['security_score']:.0f}%")
    
if "bp_score" in detailed_report:
    print(f"📐 Best Practices Score: {detailed_report['bp_score']:.0f}%")

## Étape 8: Créer un Callback Personnalisé

Vous pouvez facilement étendre le callback pour vos besoins spécifiques :

In [ ]:
from terraform_agent.callbacks import DetailedTerraformCallback

class CustomCallback(DetailedTerraformCallback):
    """Custom callback with additional checks."""
    
    def on_tool_end(self, output: str, **kwargs):
        super().on_tool_end(output, **kwargs)
        
        tool_name = kwargs.get("name", "unknown")
        
        # Add custom logic
        if tool_name == "terraform_plan":
            if "to add" in output:
                print("   💡 Resources will be created")

# Use custom callback
custom_callback = CustomCallback(verbose=True)
print("\n✓ Custom callback ready for use")

## Comparaison des Approches

### Avant (sans callback)
```python
result = generator.run(user_prompt)
# Output: logs bruts de l'agent
```

### Avec callback simple
```python
callback = TerraformPhaseCallback(verbose=True)
result = generator.run(user_prompt, callbacks=[callback])
# Output: Phases explicites + durées
```

### Avec callback détaillé
```python
callback = DetailedTerraformCallback(verbose=True)
result = generator.run(user_prompt, callbacks=[callback])
# Output: Phases + checks sécurité/BP + scores
report = callback.get_report()
```

## Avantages de l'Approche Callbacks

| Aspect | Pipeline Wrapper | Callbacks LangChain |
|--------|------------------|---------------------|
| **Architecture** | Wrapper superficiel | ✅ Natif LangChain |
| **Détection phases** | Parsing de texte | ✅ Hooks d'outils réels |
| **Intrusivité** | Nouveau composant | ✅ Option sur Generator existant |
| **Extensibilité** | Hériter du wrapper | ✅ Hériter du callback |
| **Maintenance** | Code dupliqué | ✅ Composant indépendant |
| **Performance** | Overhead | ✅ Minimal |

## Cas d'Usage

### 1. Développement/Debug
```python
callback = TerraformPhaseCallback(verbose=True)
generator.run(prompt, callbacks=[callback])
```

### 2. CI/CD avec Checks
```python
callback = DetailedTerraformCallback(verbose=False)
generator.run(prompt, callbacks=[callback])
report = callback.get_report()

if report["security_score"] < 80:
    raise Exception("Security score too low")
```

### 3. Logging Structuré
```python
import json
callback = DetailedTerraformCallback(verbose=False)
generator.run(prompt, callbacks=[callback])

# Save to file
report_file = f"logs/run-{datetime.now():%Y%m%d-%H%M%S}.json"
Path(report_file).write_text(json.dumps(callback.get_report(), indent=2))
```